# Planar point location: find the face containing a point

A point-in-polygon test answers a local question about one polygon. A planar point locator answers a broader question: which face of a subdivision contains a query coordinate?

This lab uses the repository's [static slab point locator](../implementations/python/point-location/README.md). It is deliberately not presented as a randomized trapezoidal map. The goal is to inspect x-events, open slabs, edge order and boundary behaviour in a small, reproducible subdivision.

Run from the repository root in JupyterLab or open the notebook in Colab after the branch is available.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from ipywidgets import IntSlider, interact

root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'implementations/python/point-location').is_dir()), None)
if root is None:
    raise FileNotFoundError('Run this notebook from a checkout of geo-dev-hub/spatial-analysis.')
sys.path.insert(0, str(root / 'implementations/python/point-location'))
from slab_locator import BOUNDARY, SlabPointLocator

print('Repository:', root)

## 1. Define a small planar subdivision

We use two adjacent faces and a third face containing a hole. The polygons share a boundary but their interiors do not overlap.

The locator returns a face identifier for an interior point, `None` for the unbounded/outside face, and `BOUNDARY` for a point on a represented edge. Those are three distinct outcomes.

In [ ]:
def rectangle(x1, y1, x2, y2):
    return [(x1, y1), (x2, y1), (x2, y2), (x1, y2), (x1, y1)]

faces = {
    'west': [rectangle(0, 0, 2, 4)],
    'east': [rectangle(2, 0, 4, 4)],
    'courtyard': [rectangle(5, 0, 9, 4), rectangle(6, 1, 7, 2)],
}
locator = SlabPointLocator(faces)

queries = [
    ('west interior', (1, 1), 'west'),
    ('east interior', (3, 1), 'east'),
    ('courtyard interior', (8, 3), 'courtyard'),
    ('inside the hole', (6.5, 1.5), None),
    ('shared edge', (2, 1), BOUNDARY),
    ('vertex', (0, 0), BOUNDARY),
    ('outside', (10, 2), None),
    ('event line, outside face', (2, 5), None),
]

for label, (x, y), expected in queries:
    actual = locator.locate(x, y)
    assert actual == expected, (label, expected, actual)
    print(f'{label:28} query=({x:4g}, {y:4g})  result={actual!r}')

print('\nAll expected classifications matched.')

## 2. See the vertical slabs

The locator places an x-event at each vertex x-coordinate. Between two consecutive events, the order of non-crossing edges cannot change. That lets the query search the ordered crossings for its y-position.

The vertical lines below mark the event coordinates. They are decomposition boundaries, not polygon edges.

In [ ]:
def draw_subdivision(query_index=None):
    fig, ax = plt.subplots(figsize=(11, 5))
    for face_id, rings in faces.items():
        shell = rings[0]
        ax.add_patch(Polygon(shell, closed=True, alpha=0.25, label=face_id))
        for ring in rings:
            xs, ys = zip(*ring)
            ax.plot(xs, ys, linewidth=2)
        for hole in rings[1:]:
            ax.add_patch(Polygon(hole, closed=True, facecolor='white', edgecolor='black', hatch='///'))
        cx = sum(p[0] for p in shell[:-1]) / (len(shell) - 1)
        cy = sum(p[1] for p in shell[:-1]) / (len(shell) - 1)
        ax.text(cx, cy, face_id, ha='center', va='center', weight='bold')

    for x in locator.x_events:
        ax.axvline(x, linestyle=':', linewidth=0.8, alpha=0.65)
        ax.text(x, 4.35, f'x={x:g}', rotation=90, ha='center', va='bottom', fontsize=8)

    if query_index is not None:
        label, (qx, qy), expected = queries[query_index]
        result = locator.locate(qx, qy)
        ax.scatter([qx], [qy], marker='x', s=100, linewidths=2)
        ax.annotate(f'{label}\nresult: {result!r}', (qx, qy), xytext=(8, 10),
                    textcoords='offset points', fontsize=9)

    ax.set_xlim(-0.7, 10.7)
    ax.set_ylim(-0.6, 5.0)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('x coordinate')
    ax.set_ylabel('y coordinate')
    ax.set_title('Faces, a hole and the x-event lines')
    ax.grid(True, alpha=0.2)
    plt.show()

draw_subdivision()

## 3. Inspect a query one at a time

Choose a query with the slider. Notice the difference between an interior point, a point on a shared edge, a vertex, a point inside a hole and a point outside the bounded subdivision.

Queries on x-event lines use a boundary-aware fallback in this teaching implementation. They are not answered by pretending the event line is an ordinary open slab.

In [ ]:
def inspect_query(index=0):
    label, (x, y), expected = queries[index]
    actual = locator.locate(x, y)
    draw_subdivision(index)
    print('Query:', label)
    print('Coordinate:', (x, y))
    print('Expected:', expected)
    print('Returned:', actual)
    print('On x-event:', x in locator.x_events)
    assert actual == expected

interact(inspect_query, index=IntSlider(value=0, min=0, max=len(queries)-1,
                                        step=1, description='Query'))

## 4. Look inside the slab representation

For each open x-slab, the implementation stores crossing groups in their vertical order at a midpoint. Each group records the face immediately above it after crossing the boundary.

This is a deliberately straightforward static structure. An edge can appear in many slabs, so preprocessing and storage can reach quadratic size. It should not be confused with the randomized trapezoidal-map search DAG.

In [ ]:
for i, (left, right) in enumerate(zip(locator.x_events, locator.x_events[1:])):
    groups = locator.slabs[i]
    print(f'Open slab ({left:g}, {right:g}): {len(groups)} crossing groups')
    for group in groups:
        edge, face_id = group['edges'][0]
        print(f"  y(midpoint)={group['y_mid']:.3f}; boundary face={face_id!r}; face above={group['face_after']!r}")

## 5. What this lab does—and does not—show

- It demonstrates a static x-slab point locator for valid, pairwise interior-disjoint polygon faces with linear rings.
- It demonstrates why the edge order is stable inside an open slab when the subdivision has no crossings.
- It makes boundary classification explicit.
- It does **not** construct a randomized trapezoidal map, support arbitrary intersecting arrangements or claim the expected bounds of Seidel's algorithm.

## Try these changes

1. Add a third adjacent face and predict the x-event sequence before rebuilding the locator.
2. Add a second hole to the courtyard. Test points inside and outside both holes.
3. Add a query on a horizontal edge, a vertical edge and a shared vertex.
4. Deliberately overlap two polygon interiors. Observe why the locator rejects the subdivision.
5. Explain why sorting all edges for every query would not be the same as preprocessing a point-location structure.

## Continue

- [Point-location lesson](../algorithms/geometry/point-location.md)
- [Static slab locator implementation](../implementations/python/point-location/README.md)
- [Core GIS continuation exercises](../exercises/core-gis-continuation.md)
- [Seidel (1991): randomized trapezoidal decomposition](https://doi.org/10.1016/0925-7721(91)90012-4)
